# V2: Vocabulary and Min_Frequency

## 3.1 Analyzing Token Frequency
- Goal: Determine the minimum threshold: 
- Conclusion: min_freq = 4

In [1]:
import pandas as pd
from collections import Counter
train_v2_token = pd.read_parquet('data/processed/train_v2_tokenized.parquet')

token_frequency = Counter()
for tokens in train_v2_token['tokens']:
    token_frequency.update(set(tokens))


filtered_counter = Counter({word: count for word, count in token_frequency.items() if count == 4})
print(filtered_counter.most_common(5))
print(f"Number of Tokens: {len(token_frequency)}")
print(f"Number of Filtered Tokens: {len(filtered_counter)}")


[('checker', 4), ('alice', 4), ('node', 4), ('taxi', 4), ('"win"', 4)]
Number of Tokens: 5144
Number of Filtered Tokens: 137


In [2]:
initial_total = len(token_frequency)
validation_v2 = pd.read_parquet("data/processed/validation_v2_tokenized.parquet")

validation_token_frequency = Counter()
for tokens in validation_v2['tokens']:
    validation_token_frequency.update(tokens)
total_validation_tokens = sum(validation_token_frequency.values())


## 3.2 - Looking at the Validation OOV Rate
- Validation Out of Vocaublary Rate
- Formula: Absent / All Validation Tokens
- I decided to choose `min_frequency = 4` vs choosing a `min_frequency = 3`. On the surface, choosing a min_freq of 3 seems great because it is able to remove 77.02% of rare rokens while maintaining a `1.89` OOV. Consider that the baseline (`min_freq = 0`)for the OOV was already 1.16 percent. By only increasing the Validation OOV Rate by only .15%, we are able to remove roughly 20% more than min_freq, while maintaining 97.96% of validation token occurrences.

In [3]:
print(f'Initial Starting Unique Token Amount: {initial_total}\n')
for min_freq in range(1, 6):
    print(f'min_freq = {min_freq}')

    removed_tokens_train = {word: count for word, count in token_frequency.items() if count < min_freq}
    removed_from_train = len(removed_tokens_train)
    vocab_size = initial_total - removed_from_train
    retained_vocab = {word: count for word, count in token_frequency.items() if count >= min_freq}
    out_of_vocab = 0

    for tokens in validation_v2['tokens']:
        for token in tokens:
            if token not in retained_vocab:
                out_of_vocab += 1

    print(f'Vocab Size Leftover: {vocab_size}')
    print(f'Removed (freqency < {min_freq}): {removed_from_train}')
    print(f'Percentage of Removed Tokens: {(removed_from_train / initial_total) * 100 :.2f}')
    print(f'Validation OOV Rate: {(out_of_vocab / total_validation_tokens) * 100:.2f}\n')

Initial Starting Unique Token Amount: 5144

min_freq = 1
Vocab Size Leftover: 5144
Removed (freqency < 1): 0
Percentage of Removed Tokens: 0.00
Validation OOV Rate: 1.16

min_freq = 2
Vocab Size Leftover: 1819
Removed (freqency < 2): 3325
Percentage of Removed Tokens: 64.64
Validation OOV Rate: 1.62

min_freq = 3
Vocab Size Leftover: 1182
Removed (freqency < 3): 3962
Percentage of Removed Tokens: 77.02
Validation OOV Rate: 1.89

min_freq = 4
Vocab Size Leftover: 943
Removed (freqency < 4): 4201
Percentage of Removed Tokens: 81.67
Validation OOV Rate: 2.04

min_freq = 5
Vocab Size Leftover: 806
Removed (freqency < 5): 4338
Percentage of Removed Tokens: 84.33
Validation OOV Rate: 2.19



## 3.3 - Investigating Rows with Comments
- Some of the rows had comments so in order to decide on how to move forward (remove or keep), I need to look at how important these comments are to the code
- Looked at rows with only comments to see if they were used for lables to determine if we can remove the comment tokens
- Conclusion: None of the rows in the examples were made up purely of just comments. The comment tokens that were found contribute to over 1,600 occurances consisting primarily of debug statements, notebook artfacts, encoding declarations, and natural language comments. Manual inspection showed that the five target labels were determined by the changes to the executable Python code rather than the comment contents. Comment tokens were tremoved from the tokenizer to reduce vocabulary size, while maintaining the represenation of executable code for the label.

In [4]:
from src.v2.comment_only_diff import is_comment_only

filtered_contains_comments = train_v2_token[train_v2_token['diff'].str.contains('#')]
print(f'Amount of rows that contain comments: {len(filtered_contains_comments)}')
for i in [1, 2, 3, 10, 20, 30, 50, 100, 150, 200, 300]:
    print(f"{i} ---- {filtered_contains_comments['diff'].iloc[i]} \n Top Level Label: {filtered_contains_comments['top_level_label'].iloc[i]}")

#Goal: return all of the lines that have comments only so this returns true
filtered_only_comments = train_v2_token.copy()
filtered_only_comments['only_comments'] = train_v2_token['diff'].apply(is_comment_only)
display(filtered_only_comments[filtered_only_comments['only_comments'] == True])

#Goal: find all of the unique comments strings in diff
unique_comments = Counter()
for row in filtered_contains_comments['diff']:
    lines = row.splitlines()
    for line in lines:
        line = line.replace(line[0], '', 1)
        line = line.strip()
        if not line:
            continue
        elif line.startswith('#'):
            unique_comments[line] += 1

for comment in unique_comments.most_common(10):
    print(comment)

print(len(unique_comments))
    

Amount of rows that contain comments: 1471
1 ---- - print(a)
+ #print(a) 
 Top Level Label: call
2 ---- - print(t)
+ # print(t) 
 Top Level Label: call
3 ---- - print(dp)
+ #print(dp) 
 Top Level Label: call
10 ---- + #162a
-     print(top_keta)
+     #print(top_keta) 
 Top Level Label: call
20 ---- - 		print(i)
+ 		#print(i) 
 Top Level Label: call
30 ---- -         print(sum_list)
+         # print(sum_list) 
 Top Level Label: call
50 ---- -       if s[i - 1] == s[j - 1] and LCSRe[i - 1][j - 1] < (j - i): 
+       if s[i - 1] == s[j - 1] and LCSRe[i - 1][j - 1] < j - i: 
-         else: 
+       else: 
-           LCSRe[i][j] = 0
+         LCSRe[i][j] = 0
+   #print(LCSRe) 
 Top Level Label: control_flow
100 ---- -     print(v)
+ # print(f"{pi} {qi}")
+ 
+  
 Top Level Label: call
150 ---- -   print(s)
+   #print(s) 
 Top Level Label: call
200 ---- - print(ans)
+ #print(ans) 
 Top Level Label: call
300 ---- - print(l)
+ #print(l) 
 Top Level Label: call


,diff,top_level_label,tokens,only_comments


('#print(s)', 30)
('#print(i)', 29)
('#print(a)', 20)
('#print(ans)', 19)
('#print(l)', 18)
('#print(dp)', 16)
("# '11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111

## 3.4 Investigating Various Rows For Determining Minimum Threshold
- I saw tokens like `10000000000000` and I wanted to see it in context
- Upon inspecting an example, I found a top label of `call` where it had lots of indendation surrounding it. This could be potential noise for the model so I decided to change the tokenizer to reflect the indentation rather than just removing it.

In [5]:
contains_token = train_v2_token[train_v2_token['diff'].str.contains('gcds')]

print(len(contains_token['diff']))
for i in range(1):
    print(f'Top Level Label: {contains_token['top_level_label'].iloc[i]}')
    print(contains_token['diff'].iloc[i])
    print('------')


1
Top Level Label: expression
+ 
- for i in range(n + 1):
+ for i in range(n):
- 	gcds = max(gcds,(math.gcd(r[i],l[n - i])))
+ 	gcds = max(gcds,(math.gcd(r[i],l[n - i - 1])))
------


## 3.5 - Investigating the Indents Token
- This helped me decide whether or not I needed to keep certain indent tokens. I decided to keep them because they are useful context for the model to distinguish between indentation changes and real changes.

In [6]:
indents = {token: count for token, count in token_frequency.items() if token.startswith('<INDENT_')}
print(indents)


{'<INDENT_0>': 14887, '<INDENT_1>': 6149, '<INDENT_3>': 971, '<INDENT_2>': 2702, '<INDENT_4>': 261, '<INDENT_5>': 60, '<INDENT_6>': 12, '<INDENT_7>': 1, '<INDENT_9>': 1}


## 3.6 - Applying the `min_freq` and maintaining created Markers

In [7]:
vocabulary_after_applying_min = {token: count for token, count in token_frequency.items() if count >= 4 or token in ['<ADD>', '<DELETE>'] or token.startswith('<INDENT_')}
print(f'Size of Vocabulary: {len(vocabulary_after_applying_min)}')

Size of Vocabulary: 945


## 3.7 - Building the Vocabulary

In [8]:
from src.v2.ids_and_tokens import create_token_to_id, create_id_to_token
import json
final_retained_vocab = {word: count for word, count in token_frequency.items() if count >= 4 or word.startswith('<INDENT_') or word in ('<ADD>', '<DELETE')} #kept all indents
token_to_id = create_token_to_id(final_retained_vocab)
id_to_token = create_id_to_token(token_to_id)

with open('data/processed/token_to_id.json', 'w', encoding='utf-8') as file:
    json.dump(token_to_id, file, indent=2, ensure_ascii=False)

with open('data/processed/id_to_token.json', 'w', encoding='utf-8') as file:
    json.dump(id_to_token, file, indent=2, ensure_ascii=False)

print(f'Amount of tokens after min_freq: {len(final_retained_vocab)}')
print(f'Amount of tokens after token to id: {len(token_to_id)}') #should have +2 (<PAD> and <UNK>)


Amount of tokens after min_freq: 945
Amount of tokens after token to id: 947


## 3.8 Applying the `tokens_to_ids()` 

In [ ]:
from src.v2.ids_and_tokens import tokens_to_ids
train_v2_token['tokens_ids'] = train_v2_token['tokens'].apply(lambda row_tokens: tokens_to_ids(row_tokens, token_to_id))

validation_v2['tokens_ids'] = validation_v2['tokens'].apply(lambda row_tokens: tokens_to_ids(row_tokens, token_to_id))

train_v2_token.to_parquet('data/processed/train_v2_numericalized.parquet', index=False)
validation_v2.to_parquet('data/processed/validation_v2_numericalized.parquet', index=False)

,diff,top_level_label,tokens,tokens_ids
0,"- for i, j in enumerate(itertools.permutations...",call,"[<DELETE>, <INDENT_0>, for, i, ,, j, in, enume...","[248, 249, 542, 576, 161, 603, 583, 521, 154, ..."
1,"- a,b=map(int,input())\n+ a,b=map(int,input()....",call,"[<DELETE>, <INDENT_0>, a, ,, b, =, map, (, int...","[248, 249, 337, 161, 388, 258, 662, 154, 592, ..."
2,- new += (new + v[i])/2\n+ new = (new ...,assignment,"[<DELETE>, <INDENT_1>, new, +=, (, new, +, v, ...","[248, 250, 701, 160, 154, 701, 159, 908, 332, ..."
3,+ #with open('random_pm00.txt') as f:\n+ #...,call,"[<DELETE>, <INDENT_0>, print, (, n, ,, k, )]","[248, 249, 760, 154, 693, 161, 606, 155]"
4,- while i < 5:\n+ while True:,control_flow,"[<DELETE>, <INDENT_0>, while, i, <, 5, :, <ADD...","[248, 249, 920, 576, 244, 223, 242, 247, 249, ..."
...,...,...,...,...
22804,"- print(max(dp1[n-1], dp2[n], dp3[n]))\n+ ...",call,"[<DELETE>, <INDENT_1>, print, (, max, (, dp1, ...","[248, 250, 760, 154, 665, 154, 509, 332, 693, ..."
22805,- b.pop(b[0])\n+ b.pop(0)\n-,call,"[<DELETE>, <INDENT_0>, b, ., pop, (, b, [, 0, ...","[248, 249, 388, 164, 750, 154, 388, 332, 170, ..."
22806,- ans=0\n- for i in range(n+1):\n+ for i in ra...,expression,"[<DELETE>, <INDENT_0>, ans, =, 0, <DELETE>, <I...","[248, 249, 367, 258, 170, 248, 249, 542, 576, ..."
22807,- a = list(int(input()) for i in range(n))\n+ ...,assignment,"[<DELETE>, <INDENT_0>, a, =, list, (, int, (, ...","[248, 249, 337, 258, 640, 154, 592, 154, 588, ..."


## 3.9 Validation for Both Train and Validation Sets

In [10]:
#Checking to see if there are <PAD> tokens
pads_train = train_v2_token['tokens'].map(lambda tokens: '<PAD>' in tokens)
print(pads_train[pads_train])
pads_validation = validation_v2['tokens'].map(lambda tokens: '<PAD>' in tokens)
print(pads_validation[pads_validation])

#Verifying that both lengths match (token and id)
print(f'Do the rows match? (train): {(train_v2_token['tokens'].apply(len) == train_v2_token['tokens_ids'].apply(len)).all()}')
print(f'Do the rows match? (validation): {(validation_v2['tokens'].apply(len) == validation_v2['tokens_ids'].apply(len)).all()}')

#Verfying all the rows are in the vocab range:
last_id = len(token_to_id) - 1
print(f'Valid ID range: 0 - {last_id}')
valid_ids_train = train_v2_token['tokens_ids'].apply(lambda token_ids: all(isinstance(token_id, int) and 0 <= token_id <= last_id for token_id in token_ids)).all()
valid_ids_validation = validation_v2['tokens_ids'].apply(lambda token_ids: all(isinstance(token_id, int) and 0 <= token_id <= last_id for token_id in token_ids)).all()

print(f'Are all rows valid (train): {valid_ids_train}')
print(f'Are all rows valid (validation): {valid_ids_validation}')

#Counting the amount of UNK IDs
unknown_id = token_to_id['<UNK>']
train_unknowns = train_v2_token['tokens_ids'].apply(lambda tokens: sum(token_id == unknown_id for token_id in tokens)).sum()
validation_unknowns = validation_v2['tokens_ids'].apply(lambda tokens: sum(token_id == unknown_id for token_id in tokens)).sum()

print(f'Unknowns in train: {train_unknowns}')
print(f'Unknowns in validation: {validation_unknowns}')

# Counting the amount of total tokens
total_train_ids = train_v2_token['tokens_ids'].apply(len).sum()
train_unknown_rate = round((train_unknowns / total_train_ids) * 100, 4)
print(f'Train Unknown Rate: {train_unknown_rate}')

total_validation_ids = validation_v2['tokens_ids'].apply(len).sum()
validation_unknown_rate = round((validation_unknowns / total_validation_ids) * 100, 4)
print(f'Validation Unknown Rate: {validation_unknown_rate}')

#Checking that there are no padding_ids in neither split
padding_id = token_to_id['<PAD>']
assert not train_v2_token['tokens_ids'].map(lambda token_ids: padding_id in token_ids).any()
assert not validation_v2['tokens_ids'].map(lambda token_ids: padding_id in token_ids).any()


Series([], Name: tokens, dtype: bool)
Series([], Name: tokens, dtype: bool)
Do the rows match? (train): True
Do the rows match? (validation): True
Valid ID range: 0 - 946
Are all rows valid (train): True
Are all rows valid (validation): True
Unknowns in train: 10861
Unknowns in validation: 2967
Train Unknown Rate: 1.8652
Validation Unknown Rate: 2.0405


## 3.10 Investigating Padding:

In [11]:
train_sequence_lengths = train_v2_token['tokens_ids'].apply(len)
print(train_sequence_lengths.describe(percentiles= [.90, .95, .99]))
print(f"Median: {train_sequence_lengths.median()}")

count    22809.000000
mean        25.529221
std         24.316373
min          3.000000
90%         52.000000
95%         66.000000
99%        112.000000
max        554.000000
Name: tokens_ids, dtype: float64
Median: 21.0


In [12]:
for max_length in [52, 64, 66, 112]:
    print(f'The max length is: {max_length}')
    candidates = train_sequence_lengths[train_sequence_lengths <= max_length]
    total_rows = len(train_sequence_lengths)
    preserved_fully = len(candidates)
    truncated = total_rows - preserved_fully
    removed_tokens_rows = train_sequence_lengths[train_sequence_lengths > max_length]
    total_tokens = train_sequence_lengths.sum() #sums up all of the tokens total
    removed_tokens_count = (removed_tokens_rows - max_length).sum()
    retained_tokens = total_tokens - removed_tokens_count

    print(f'Percentage of sequences fully preserved: {preserved_fully / total_rows * 100:.2f}')
    print(f'Number of sequences truncated: {total_rows - preserved_fully}')
    print(f'Percentage of tokens retained: {retained_tokens / total_tokens * 100:.2f}')
    print(f'Number of tokens removed: {removed_tokens_count}\n')

The max length is: 52
Percentage of sequences fully preserved: 90.68
Number of sequences truncated: 2126
Percentage of tokens retained: 89.90
Number of tokens removed: 58823

The max length is: 64
Percentage of sequences fully preserved: 94.67
Number of sequences truncated: 1215
Percentage of tokens retained: 93.38
Number of tokens removed: 38559

The max length is: 66
Percentage of sequences fully preserved: 95.11
Number of sequences truncated: 1116
Percentage of tokens retained: 93.79
Number of tokens removed: 36140

The max length is: 112
Percentage of sequences fully preserved: 99.03
Number of sequences truncated: 222
Percentage of tokens retained: 98.05
Number of tokens removed: 11353



In [13]:
long_sequences = train_sequence_lengths[train_sequence_lengths > 64]
example_index = long_sequences.idxmin()

print("Index:", example_index)
print("Length:", train_sequence_lengths.loc[example_index])
print("Diff:", train_v2_token.loc[example_index, "diff"])
print("Label:", train_v2_token.loc[example_index, "top_level_label"])

example_tokens = train_v2_token.loc[example_index, "tokens"]

print("Last 10 retained tokens:", example_tokens[:64][-10:])
print("Removed tokens:", example_tokens[64:])

Index: 702
Length: 65
Diff: -     if n[0] == [1] == n[2] or n[1] == n[2] == n[3]:
+     if n[0] == n[1] == n[2] or n[1] == n[2] == n[3]:
Label: control_flow
Last 10 retained tokens: ['==' 'n' '[' '2' ']' '==' 'n' '[' '3' ']']
Removed tokens: [':']


In [14]:
target_length = 150

example_index = (train_sequence_lengths[train_sequence_lengths > 64].sub(target_length).abs().idxmin())

print('Index:', example_index)
print('Length:', train_sequence_lengths.loc[example_index])
print('Diff:', train_v2_token.loc[example_index, 'diff'])
print('Label:', train_v2_token.loc[example_index, 'top_level_label'])

example_tokens = train_v2_token.loc[example_index, 'tokens']
print('Last 10 retained tokens:', example_tokens[:64][-10:])
print('Removed tokens:', example_tokens[64:])

Index: 14208
Length: 150
Diff: -   if l[i][1]-l[i][0]!=l[i+1][1]-l[i+1][0] and l[i][2]-l[i][1]!=l[i+1][2]-l[i+1][1]:
+   if l[i][1]-l[i][0]!=l[i+1][1]-l[i+1][0] or l[i][2]-l[i][1]!=l[i+1][2]-l[i+1][1]:
Label: control_flow
Last 10 retained tokens: ['!=' 'l' '[' 'i' '+' '1' ']' '[' '2' ']']
Removed tokens: ['-' 'l' '[' 'i' '+' '1' ']' '[' '1' ']' ':' '<ADD>' '<INDENT_0>' 'if' 'l'
 '[' 'i' ']' '[' '1' ']' '-' 'l' '[' 'i' ']' '[' '0' ']' '!=' 'l' '[' 'i'
 '+' '1' ']' '[' '1' ']' '-' 'l' '[' 'i' '+' '1' ']' '[' '0' ']' 'or' 'l'
 '[' 'i' ']' '[' '2' ']' '-' 'l' '[' 'i' ']' '[' '1' ']' '!=' 'l' '[' 'i'
 '+' '1' ']' '[' '2' ']' '-' 'l' '[' 'i' '+' '1' ']' '[' '1' ']' ':']


## 3.11 Investigating Sequences > 112 Tokens

In [15]:
greater_than_112_tokens = train_sequence_lengths[train_sequence_lengths > 112]
long_rows = len(greater_than_112_tokens)
print(f'Amount of rows (> 112 tokens): {long_rows}')


valid_rows = train_v2_token[train_v2_token['tokens_ids'].apply(len) > 112]
long_counts = Counter(valid_rows['top_level_label'])
total_counts = Counter(train_v2_token['top_level_label'])

for label, amount in long_counts.items():
    print(f'Label: {label}\nAmount of Rows: {amount}\nPercentage of all long rows: {amount / long_rows * 100:.2f}%\n')

for label, total_amount in total_counts.items():
    long_amount = long_counts[label]
    print(f'Percentage of {label}s that are long: {long_amount / total_amount * 100:.2f}%')




Amount of rows (> 112 tokens): 222
Label: call
Amount of Rows: 65
Percentage of all long rows: 29.28%

Label: assignment
Amount of Rows: 33
Percentage of all long rows: 14.86%

Label: control_flow
Amount of Rows: 62
Percentage of all long rows: 27.93%

Label: expression
Amount of Rows: 51
Percentage of all long rows: 22.97%

Label: identifier
Amount of Rows: 11
Percentage of all long rows: 4.95%

Percentage of calls that are long: 0.59%
Percentage of assignments that are long: 1.12%
Percentage of control_flows that are long: 1.80%
Percentage of expressions that are long: 1.07%
Percentage of identifiers that are long: 1.59%


In [16]:
only_control_flow_rows = train_v2_token[(train_v2_token['top_level_label'] == 'control_flow') & (train_v2_token['tokens'].map(len) > 112)]
sorted_by_sequence_len = only_control_flow_rows.assign(sequence_len=only_control_flow_rows['tokens'].map(len)).sort_values('sequence_len')
median = sorted_by_sequence_len.describe().loc['50%'].loc['sequence_len']
longest = sorted_by_sequence_len.describe().loc['max'].loc['sequence_len']

shortest_row_index = int(sorted_by_sequence_len['sequence_len'].sub(112).abs().idxmin())
median_row_index = int(sorted_by_sequence_len['sequence_len'].sub(median).abs().idxmin())
longest_row_index = int(sorted_by_sequence_len['sequence_len'].sub(longest).abs().idxmin())

for index in [shortest_row_index, median_row_index, longest_row_index]:
    row = sorted_by_sequence_len.loc[index]
    cut_tokens = sorted_by_sequence_len['tokens'].loc[index][112:]
    print(f'Length: {row['sequence_len']}')
    print(f'Cut tokens: {cut_tokens}')
    print(f'Code:\n{row['diff']}\n')
    

Length: 114
Cut tokens: ['g_s' ':']
Code:
- k_s = set(list(s[0::2]))#奇数
+ k_s = set(list(s[0::2]))
- g_s = set(list(s[1::2]))#偶数
+ g_s = set(list(s[1::2]))
- 
- 
- if {"R","U","D"} >= k_s or {"L","U","D"} >= g_s:
+ if {"R","U","D"} >= k_s and {"L","U","D"} >= g_s:
- 

Length: 145
Cut tokens: ['+' 'j' ':' ']' '<ADD>' '<INDENT_5>' 'new' '=' 's' '[' ':' 'start' '+'
 'j' ']' '+' 't' '+' 's' '[' 'start' '+' 'len' '(' 't' ')' '+' 'j' ':' ']'
 '<DELETE>' '<INDENT_5>' 'break']
Code:
-             for j in range(count - len(t) + 1):
+             for j in range(count-len(t)+1):
-                     if s[start + j + k] != t[k] and s[start + j + k] != '?':
+                     if s[start+j+k] != t[k] and s[start+j+k] != '?':
-                     new = s[:start + j] + t + s[start + len(t) + j:]
+                     new = s[:start+j] + t + s[start+len(t)+j:]
-                     break

Length: 487
Cut tokens: ['-' 'f' ')' ')' '==' '0' ':' '<ADD>' '<INDENT_0>' 'if' '(' '2' '*' '('
 'e' '-' 'a' 

## Conclusion

The vocabulary was built using only the training data to prevent validation leakage. Token frequency was measured using document frequency, where each document represents one tokenized diff. A `min_freq` of 4 was used to exclude uncommon token types, while rare indentation tokens were protected because they preserve structural information.

`<PAD>` and `<UNK>` were assigned IDs `0` and `1`, respectively. The final vocabulary contained 947 entries. Although 81.67% of the unique training token types fell below the frequency threshold, the validation OOV rate remained low at 2.04%. This showed that most of the excluded token types were uncommon and did not represent many actual token occurrences.

The training and validation token sequences were then converted into integer IDs. Validation had a slightly higher `<UNK>` rate than training, which was expected because the vocabulary was built using only the training data.

I initially chose 64 as the maximum sequence length based on the sequence-length statistics. After investigating some of the longer rows, I realized that a limit of 64 would not always preserve the actual fix. The important change could appear at the beginning, middle, or end of the diff. In some examples, head truncation preserved formatting changes while removing the change responsible for the label, such as removing a `break` statement or adding a numerical tolerance at the end.

Because of this, I decided to keep every sequence without truncation. During model training, I will use dynamic padding and length-aware batching to reduce padding waste while preserving all of the original tokens. If keeping every token causes an actual memory or runtime problem during training, I can reconsider this decision using the results from those experiments.
